# American Option Pricing with Longstaff-Schwartz Method (LSM)

### Black & Scholes

In [1]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

### CRR Binominal Model

In [2]:
def binomial(S, K, T, r, sigma, n, option_type='call', exercise_type='european', model='CRR'):
    dt = T / n
    df = np.exp(-r * dt)
    sign = 1 if option_type == 'call' else -1
    if model == 'CRR':
        u = np.exp(sigma * np.sqrt(dt))
        d = 1 / u
        q = (np.exp(r * dt) - d) / (u - d)
    
    # 1. Initialize terminal payoffs
    prices = S * (u**np.arange(n, -1, -1)) * (d**np.arange(n + 1))
    values = np.maximum(sign * (prices - K), 0)
    
    # 2. Backward induction
    for i in range(n - 1, -1, -1):
        values = df * (q * values[:-1] + (1 - q) * values[1:])
        
        if exercise_type == 'american':
            prices = prices[:-1] / u 
            exercise = np.maximum(sign * (prices - K), 0)
            values = np.maximum(exercise, values)
        
    return values[0]

### Longstaff-Schwartz Method (LSM)

In [3]:
def simulate_paths(S, T, r, sigma, n_step, n_sim , seed=42):
    np.random.seed(seed)
    dt = T / n_step
    
    # Generate a matrix of Z ~ N(0, 1) with dimensions (nb of scenarios, nb of steps)
    Z = np.random.standard_normal((n_sim, n_step))
    growth = np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    # Paths matrix dimension (nb of scenarios, nb of steps + 1)
    paths = np.ones((n_sim, n_step + 1)) * S
    paths[:, 1:] = S * np.cumprod(growth, axis=1)
    
    return paths

In [4]:
def lsm_american(S, K, T, r, sigma, n_step, n_sim, option_type='call', degree=2, seed=42):
    paths = simulate_paths(S, T, r, sigma, n_step, n_sim, seed)
    sign = 1 if option_type == 'call' else -1
    disc = np.exp(-r * T / n_step)
    
    def payoff(s):
        return np.maximum(sign * (s - K), 0)

    # --- 1. Payoff at maturity ---
    n_paths = paths.shape[0]
    cashflow = payoff(paths[:, -1]).copy()      
    exercise_time = np.full(n_paths, n_step) # time index (in steps)

    # --- 2. Backward induction with regression ---
    for t in range(n_step - 1, 0, -1):
        St = paths[:, t]
        itm = payoff(St) > 0 # only regress on ITM paths
        if itm.sum() == 0:
            continue

        # Discount currently-recorded cashflow back to time t
        Y = cashflow[itm] * disc**(exercise_time[itm] - t)
        X = St[itm]
        A = np.vstack([X**p for p in range(degree + 1)]).T # basis functions
        coef, *_ = np.linalg.lstsq(A, Y, rcond=None) # formula (14.4)
        continuation_val = A.dot(coef) # formula (14.3)

        exercise_val = payoff(St[itm])
        exercise_now = exercise_val > continuation_val # formula (14.5)

        idx_itm = np.where(itm)[0]
        idx_exercise = idx_itm[exercise_now]
        cashflow[idx_exercise] = exercise_val[exercise_now] # overwrite: exercise now
        exercise_time[idx_exercise] = t # ...at time t

    # --- 3. Discount every path's realized cashflow back to time 0 ---
    pv = cashflow * disc**exercise_time
    price = pv.mean()
    se = pv.std(ddof=1) / np.sqrt(n_paths)
    return price, se

In [5]:
# Parameters
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_step, n_sim = 252, 100000

price_lsm, se_lsm = lsm_american(S, K, T, r, sigma, n_step, n_sim, 'put', degree=2)
price_bm = binomial(S, K, T, r, sigma, 1000, 'put', 'american')
price_bs = black_scholes(S, K, T, r, sigma, 'put')

print(f"Black-Scholes                    : {price_bs:.4f}")
print(f"Binomial American (CRR, n=1000)  : {price_bm:.4f}")
print(f"LSM American (n={n_sim:}, m={n_step})   : {price_lsm:.4f} ± {1.96*se_lsm:.4f}")

Black-Scholes                    : 5.5735
Binomial American (CRR, n=1000)  : 6.0896
LSM American (n=100000, m=252)   : 6.0725 ± 0.0439


In [6]:
for deg in [1, 2, 3, 4]:
    p, se = lsm_american(S, K, T, r, sigma, n_step, n_sim, 'put', degree=deg)
    print(f"degree={deg}: price={p:.4f} ± {1.96*se:.4f}")

degree=1: price=5.9532 ± 0.0454
degree=2: price=6.0725 ± 0.0439
degree=3: price=6.0867 ± 0.0440
degree=4: price=6.0891 ± 0.0440


In [7]:
for n_simt in [1000, 5000, 10000, 100000]:
    p, se = lsm_american(S, K, T, r, sigma, n_step, n_simt, 'put', degree=2)
    print(f"n_sim={n_simt}: price={p:.4f} ± {1.96*se:.4f}")

n_sim=1000: price=6.1242 ± 0.4147
n_sim=5000: price=6.1534 ± 0.1965
n_sim=10000: price=6.1329 ± 0.1384
n_sim=100000: price=6.0725 ± 0.0439
